In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q datasets transformers
!pip install umap-learn

In [ ]:
!pip install -q huggingface_hub pandas torch

from huggingface_hub import snapshot_download
import pandas as pd
import glob
import os
import re
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Download the raw CSV files directly (matching the 'data/' folder structure you found)
repo_path = snapshot_download(
    repo_id="BreathSense/BreathSense",
    repo_type="dataset",
    allow_patterns="data/**/*drmusic.csv"
)

# 2. Locate and process files into a dictionary
all_files = glob.glob(f"{repo_path}/data/**/*drmusic.csv", recursive=True)
data_dict = {}

for file_path in all_files:
    activity = os.path.basename(os.path.dirname(file_path))
    filename = os.path.basename(file_path)
    
    # Extract the p1, p2, etc. identifier
    match = re.search(r'(p\d+)', filename, re.IGNORECASE)
    if match:
        person_id = match.group(1).lower()
    else:
        continue 
        
    df = pd.read_csv(file_path)
    sequential_data = df.to_dict(orient='records')
    
    if person_id not in data_dict:
        data_dict[person_id] = {}
        
    data_dict[person_id][activity] = sequential_data

# 3. Build the 108-row DataFrame
final_df = pd.DataFrame.from_dict(data_dict, orient='index')
desired_columns = ['rest', 'walk', 'run', 'stairs']
existing_columns = [col for col in desired_columns if col in final_df.columns]
final_df = final_df[existing_columns]

# Sort numerically to ensure p1 to p108 order
final_df = final_df.loc[
    sorted(final_df.index, key=lambda x: int(x.replace('p', '')))
]

print(f"Data Reshaped! Total Unique People: {len(final_df)}")

# 4. PyTorch Dataset & DataLoader Integration
class UniformBreathSenseDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Convert list of dictionaries into 2D lists
        rest_data = [list(step.values()) for step in row['rest']] if type(row['rest']) == list else []
        walk_data = [list(step.values()) for step in row['walk']] if type(row['walk']) == list else []
        run_data  = [list(step.values()) for step in row['run']] if type(row['run']) == list else []
        stairs_data = [list(step.values()) for step in row['stairs']] if type(row['stairs']) == list else []
        
        # Convert to tensors
        rest_tensor = torch.tensor(rest_data).float()
        walk_tensor = torch.tensor(walk_data).float()
        run_tensor  = torch.tensor(run_data).float()
        stairs_tensor = torch.tensor(stairs_data).float()
        
        return rest_tensor, walk_tensor, run_tensor, stairs_tensor

# Initialize your final DataLoader
train_dataset = UniformBreathSenseDataset(final_df)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Verify the tensor shapes output by the DataLoader
for rest, walk, run, stairs in train_loader:
    print(f"Batch Ready! Rest Tensor Shape: {rest.shape}")
    break # Stops after printing the first batch

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture # Added GMM import
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# 1. Grab the entire dataset in one single batch (108 people)
# We set shuffle=False to keep p1 to p108 in perfect order
full_loader = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=False)
rest, walk, run, stairs = next(iter(full_loader))

# Convert PyTorch tensors to Numpy arrays and remove the empty last dimension
# Shape goes from (108, 300, 1) to (108, 300)
rest_np = rest.squeeze(-1).numpy()
walk_np = walk.squeeze(-1).numpy()
run_np = run.squeeze(-1).numpy()
stairs_np = stairs.squeeze(-1).numpy()

# 2. The "Shallow" Feature Engineering Step
# We extract Mean, Standard Deviation, Max, and Min for every person's 300 steps
def extract_stats(data):
    means = np.mean(data, axis=1)
    stds = np.std(data, axis=1)
    maxes = np.max(data, axis=1)
    mins = np.min(data, axis=1)
    # Combine into a single matrix of features
    return np.column_stack((means, stds, maxes, mins))

print("Extracting statistical features...")
f_rest = extract_stats(rest_np)
f_walk = extract_stats(walk_np)
f_run = extract_stats(run_np)
f_stairs = extract_stats(stairs_np)

# Glue all 4 activities together. 
# 4 features * 4 activities = 16 total features per person
X_shallow = np.hstack((f_rest, f_walk, f_run, f_stairs))
print(f"Shallow Feature Matrix Shape: {X_shallow.shape}") # Should be (108, 16)

# 3. Scaling (Crucial for traditional K-Means and GMM)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_shallow)

# 4a. K-Means Clustering
kmeans_shallow = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_labels = kmeans_shallow.fit_predict(X_scaled)

# 4b. GMM Clustering
gmm_shallow = GaussianMixture(n_components=2, random_state=42, n_init=10)
gmm_labels = gmm_shallow.fit_predict(X_scaled)

# Add both to your dataframe to compare later
# (Assuming final_df is defined earlier in your script)
final_df['Shallow_KMeans_Cluster'] = kmeans_labels
final_df['Shallow_GMM_Cluster'] = gmm_labels

In [ ]:
import umap
import matplotlib.pyplot as plt
import seaborn as sns

print("Running UMAP dimensionality reduction...")
# 1. Initialize and fit UMAP
# We project the 16-dimensional scaled data down to 2 dimensions for visualization
reducer = umap.UMAP(n_components=2, random_state=42)
embedding = reducer.fit_transform(X_scaled)

# 2. Append UMAP coordinates to your dataframe
final_df['UMAP_1'] = embedding[:, 0]
final_df['UMAP_2'] = embedding[:, 1]

# 3. Create side-by-side scatter plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: K-Means
sns.scatterplot(
    data=final_df,
    x='UMAP_1', 
    y='UMAP_2',
    hue='Shallow_KMeans_Cluster',
    palette='Set1',  # Categorical color palette
    ax=axes[0],
    s=80,            # Marker size
    alpha=0.8        # Transparency
)
axes[0].set_title('UMAP Projection: K-Means Clusters')

# Subplot 2: GMM
sns.scatterplot(
    data=final_df,
    x='UMAP_1', 
    y='UMAP_2',
    hue='Shallow_GMM_Cluster',
    palette='Set2',  
    ax=axes[1],
    s=80,
    alpha=0.8
)
axes[1].set_title('UMAP Projection: GMM Clusters')

plt.tight_layout()
plt.show()

In [ ]:
import torch
from torch.utils.data import Subset, DataLoader

# 1. Find the literal row numbers for anyone where the ID number is <= 90
male_indices = []

for row_number, person_id in enumerate(final_df.index):
    # Extract the integer from the string (e.g., 'p45' becomes 45)
    id_number = int(person_id.replace('p', ''))
    
    # If they are p1 through p90, save their row number
    if id_number <= 90:
        male_indices.append(row_number)

print(f"Total Male Participants Found: {len(male_indices)}") 
# This should print 90!

# 2. Filter your master DataFrame so we can save the labels later
# .iloc[] isolates rows based on their numerical position
male_df = final_df.iloc[male_indices].copy()

# 3. Create the PyTorch Subset using those exact same row numbers
male_dataset = Subset(train_dataset, male_indices)

# 4. Create new DataLoaders exclusively for the male data
male_train_loader = DataLoader(male_dataset, batch_size=16, shuffle=True)
male_extract_loader = DataLoader(male_dataset, batch_size=16, shuffle=False)

In [ ]:
import pandas as pd

# Iterate through each of the 3 clusters (0, 1, and 2)
for cluster_id in range(3):
    # Extract the index (the p1, p2, p3 labels) for everyone in this specific cluster
    people = final_df[final_df['Shallow_Cluster'] == cluster_id].index.tolist()
    
    # Print the results cleanly
    print(f"========== CLASS {cluster_id} ==========")
    print(f"Total Participants: {len(people)}")
    print(f"Participants: {', '.join(people)}")
    print("\n") # Adds a blank line for readability